# Training YOLO11 with Roboflow Dataset

Notebook นี้อ้างอิงจากตัวอย่าง `Training_YOLOv8_with_Thai_Coins_Dataset.ipynb`
แต่ปรับให้เป็น workflow สำหรับ **YOLO11** และเพิ่มส่วนรายงานผลตามที่ต้องใช้ส่งงาน

หมายเหตุสำคัญ:
- URL Dataset ที่ให้มาชี้ไปยังชุดข้อมูล `Extra-Fruit&Vegetable-PromaxPlus-(Merge3Projects)` จาก Roboflow
- Dataset นี้มี **63 classes** และในไฟล์ export ที่ดาวน์โหลดได้จริงมีรูปอยู่เฉพาะ `train/`
- Notebook นี้จึงมีขั้นตอน **auto-split** เพื่อสร้าง `train / valid / test` ให้พร้อมก่อนเริ่ม train


In [1]:
!nvidia-smi


Mon Mar 30 17:11:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   43C    P0             45W /  285W |    1352MiB /  12282MiB |      6%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Workflow

1. ดาวน์โหลด dataset จาก Roboflow URL ที่ผู้ใช้ให้มา
2. ตรวจสอบโครงสร้าง dataset และ auto-split หากไม่มี `valid/test`
3. สร้าง `data.yaml` สำหรับ YOLO11
4. train โมเดลด้วย augmentation ที่กำหนด
5. ประเมินผลและสรุปค่า `mAP@50` กับ `mAP@50-95`


In [2]:
%pip install -q ultralytics pyyaml requests tqdm


Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import random
import requests
import shutil
import zipfile
from pathlib import Path

import yaml
from tqdm import tqdm
from IPython.display import Markdown, display

DATASET_URL = "https://app.roboflow.com/ds/RXD5PxVj4K?key=5yCTVODAED"
FORCE_REDOWNLOAD = False
FORCE_REEXTRACT = False

if Path("/content").exists():
    BASE_DIR = Path("/content")
else:
    BASE_DIR = Path.cwd() / "runtime_data"
    BASE_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_PATH = BASE_DIR / "roboflow_yolo11.zip"
RAW_DIR = BASE_DIR / "roboflow_raw"
WORK_DIR = BASE_DIR / "custom_data_yolo11"
SEED = 42
TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10

print("Dataset URL:", DATASET_URL)
print("Runtime base dir:", BASE_DIR)
print("Download path:", DOWNLOAD_PATH)
print("Cached zip exists:", DOWNLOAD_PATH.exists())
print("Extracted raw dataset exists:", (RAW_DIR / "data.yaml").exists())
print("Prepared split dataset exists:", (WORK_DIR / "data.yaml").exists())
print(f"Flags -> FORCE_REDOWNLOAD={FORCE_REDOWNLOAD}, FORCE_REEXTRACT={FORCE_REEXTRACT}")
print("Status: ready for download/extract step")

def download_with_progress(url: str, output_path: Path, chunk_size: int = 1024 * 1024):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    print("Status: connecting to dataset server...")

    with requests.get(url, stream=True, timeout=(30, 600), allow_redirects=True) as response:
        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))
        downloaded = 0
        last_reported_pct = -1

        if total_size > 0:
            print(f"Status: downloading dataset ({total_size / (1024 ** 3):.2f} GB)")
        else:
            print("Status: downloading dataset (size unknown)")

        with open(output_path, "wb") as file_obj, tqdm(
            total=total_size if total_size > 0 else None,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            mininterval=0.5,
            dynamic_ncols=True,
            leave=True,
            desc="Download",
        ) as progress_bar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if not chunk:
                    continue

                file_obj.write(chunk)
                size = len(chunk)
                downloaded += size
                progress_bar.update(size)

                if total_size > 0:
                    pct = int(downloaded * 100 / total_size)
                    if pct >= last_reported_pct + 10:
                        print(f"Status: {pct}% ({downloaded / (1024 ** 2):,.0f} MB / {total_size / (1024 ** 2):,.0f} MB)")
                        last_reported_pct = pct

    print(f"Status: download complete -> {output_path}")


Dataset URL: https://app.roboflow.com/ds/RXD5PxVj4K?key=5yCTVODAED
Runtime base dir: \content
Download path: \content\roboflow_yolo11.zip
Cached zip exists: True
Extracted raw dataset exists: False
Prepared split dataset exists: False
Flags -> FORCE_REDOWNLOAD=False, FORCE_REEXTRACT=False
Status: ready for download/extract step


In [8]:
if FORCE_REDOWNLOAD and DOWNLOAD_PATH.exists():
    print(f"Status: removing cached archive -> {DOWNLOAD_PATH}")
    DOWNLOAD_PATH.unlink()

if FORCE_REEXTRACT and RAW_DIR.exists():
    print(f"Status: removing extracted dataset -> {RAW_DIR}")
    shutil.rmtree(RAW_DIR)

if DOWNLOAD_PATH.exists():
    print(f"Status: using existing archive -> {DOWNLOAD_PATH}")
else:
    download_with_progress(DATASET_URL, DOWNLOAD_PATH)

if (RAW_DIR / "data.yaml").exists():
    print(f"Status: using existing extracted dataset -> {RAW_DIR}")
else:
    print("Status: extracting zip file...")
    with zipfile.ZipFile(DOWNLOAD_PATH, "r") as zip_ref:
        zip_ref.extractall(RAW_DIR)
    print("Status: extraction completed")

print("Status: download/extraction step completed")


Status: using existing archive -> \content\roboflow_yolo11.zip
Status: extracting zip file...


BadZipFile: File is not a zip file

In [9]:
missing = [name for name in ("RAW_DIR", "yaml") if name not in globals()]
if missing:
    raise RuntimeError("กรุณารัน cell 5 ก่อน cell 7; ตัวแปรที่ยังไม่มี: " + ", ".join(missing))

readme_path = RAW_DIR / "README.roboflow.txt"
data_yaml_path = RAW_DIR / "data.yaml"

if not data_yaml_path.exists():
    raise FileNotFoundError("ยังไม่พบ data.yaml ใน raw dataset กรุณารัน cell 6 ก่อน")

if readme_path.exists():
    print(readme_path.read_text(encoding="utf-8"))

raw_data = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
print("\nข้อมูลจาก data.yaml")
print(raw_data)


FileNotFoundError: ยังไม่พบ data.yaml ใน raw dataset กรุณารัน cell 6 ก่อน

## เตรียมข้อมูลและคำนวณสัดส่วน Train / Valid / Test

Cell ถัดไปจะ:
- ตรวจว่าชุดข้อมูลมี `train`, `valid`, `test` ครบหรือไม่
- ถ้าไม่ครบ จะรวมรูปทั้งหมดแล้ว split ใหม่ตามสัดส่วน `80/10/10`
- แสดงจำนวนรูปภาพและเปอร์เซ็นต์ในแต่ละ split


In [10]:
import os
import random
import shutil
from pathlib import Path

import yaml
from tqdm import tqdm
from IPython.display import Markdown, display

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLITS = ("train", "valid", "test")
PROGRESS_REPORT_EVERY = 500

missing = [
    name for name in ("RAW_DIR", "WORK_DIR", "SEED", "TRAIN_RATIO", "VALID_RATIO", "TEST_RATIO", "raw_data")
    if name not in globals()
]
if missing:
    raise RuntimeError(
        "กรุณารัน cell 5 และ cell 7 ก่อน cell 9; ตัวแปรที่ยังไม่มี: " + ", ".join(missing)
    )

def list_images(folder: Path):
    if not folder.exists():
        return []
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    )


def label_for_image(image_path: Path) -> Path:
    return image_path.parent.parent / "labels" / f"{image_path.stem}.txt"


def ensure_empty_split_dirs(base_dir: Path):
    if base_dir.exists():
        shutil.rmtree(base_dir)
    for split in SPLITS:
        (base_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (base_dir / split / "labels").mkdir(parents=True, exist_ok=True)


def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def split_counts(base_dir: Path):
    return {
        split: len(list_images(base_dir / split / "images"))
        for split in SPLITS
    }


def prepare_dataset(raw_dir: Path, work_dir: Path, raw_data: dict):
    existing_counts = split_counts(raw_dir)
    print("Status: existing split counts ->", existing_counts)
    has_complete_splits = all(existing_counts[split] > 0 for split in SPLITS)

    if has_complete_splits:
        target_dir = raw_dir
        final_counts = existing_counts
        print("พบ split ครบอยู่แล้ว ใช้ dataset เดิมได้ทันที")
    else:
        print("ไม่พบ valid/test ครบถ้วน -> สร้าง split ใหม่แบบ 80/10/10")
        ensure_empty_split_dirs(work_dir)

        all_images = []
        for split in SPLITS:
            split_folder = raw_dir / split / "images"
            split_images = list_images(split_folder)
            print(f"Status: scanned {split_folder} -> {len(split_images):,} images")
            all_images.extend(split_images)

        if not all_images:
            raise FileNotFoundError("ไม่พบไฟล์ภาพใน dataset")

        rng = random.Random(SEED)
        rng.shuffle(all_images)

        total = len(all_images)
        train_end = int(total * TRAIN_RATIO)
        valid_end = train_end + int(total * VALID_RATIO)

        mapping = {
            "train": all_images[:train_end],
            "valid": all_images[train_end:valid_end],
            "test": all_images[valid_end:],
        }

        print(f"Status: total images to prepare = {total:,}")
        print(
            "Status: target split counts -> "
            + ", ".join(f"{split}={len(images):,}" for split, images in mapping.items())
        )

        processed = 0
        with tqdm(
            total=total,
            unit="img",
            dynamic_ncols=True,
            mininterval=0.5,
            desc="Preparing dataset",
        ) as progress_bar:
            for split, images in mapping.items():
                print(f"Status: writing {split} split ({len(images):,} images)")
                for image_path in images:
                    dst_image = work_dir / split / "images" / image_path.name
                    link_or_copy(image_path, dst_image)

                    src_label = label_for_image(image_path)
                    dst_label = work_dir / split / "labels" / f"{image_path.stem}.txt"
                    if src_label.exists():
                        link_or_copy(src_label, dst_label)
                    else:
                        dst_label.touch()

                    processed += 1
                    progress_bar.update(1)
                    if processed % PROGRESS_REPORT_EVERY == 0 or processed == total:
                        print(f"Status: prepared {processed:,}/{total:,} images")

                print(f"Status: {split} split completed")

        target_dir = work_dir
        final_counts = split_counts(target_dir)
        print("Status: auto-split completed")

    prepared_yaml = {
        "path": str(target_dir),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": raw_data["nc"],
        "names": raw_data["names"],
    }

    prepared_yaml_path = target_dir / "data.yaml"
    prepared_yaml_path.write_text(
        yaml.safe_dump(prepared_yaml, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print(f"Status: wrote training yaml -> {prepared_yaml_path}")

    return target_dir, prepared_yaml_path, final_counts


prepared_dir, prepared_yaml_path, counts = prepare_dataset(RAW_DIR, WORK_DIR, raw_data)
total_images = sum(counts.values())

report_lines = [
    "## 1. สรุปจำนวนรูปภาพ",
    f"- จำนวนรูปภาพทั้งหมด: **{total_images:,}**",
    (
        f"- จำนวนรูปภาพ: Train (**{counts['train']:,} ภาพ | {counts['train'] / total_images * 100:.1f}%**) "
        f"| Valid (**{counts['valid']:,} ภาพ | {counts['valid'] / total_images * 100:.1f}%**) "
        f"| Test (**{counts['test']:,} ภาพ | {counts['test'] / total_images * 100:.1f}%**)"
    ),
    f"- จำนวนคลาส: **{raw_data['nc']} classes**",
    f"- data.yaml ที่ใช้ train: `{prepared_yaml_path}`",
]
display(Markdown("\n".join(report_lines)))


RuntimeError: กรุณารัน cell 5 และ cell 7 ก่อน cell 9; ตัวแปรที่ยังไม่มี: raw_data

## ตั้งค่าการเรียนรู้ (Training Setup)

โน้ต:
- ใช้ `yolo11s.pt` เพื่อบาลานซ์ระหว่างความเร็วและความแม่นยำ
- ถ้า Colab RAM / VRAM ไม่พอ ให้ลด `batch` จาก 16 ลงเป็น 8
- ค่า `hsv_v=0.20` ถูกใช้แทนการปรับความสว่างประมาณ `+-20%`


In [5]:
import torch
from IPython.display import Markdown, display
from ultralytics import YOLO

missing = [name for name in ("BASE_DIR", "SEED") if name not in globals()]
if missing:
    raise RuntimeError("กรุณารัน cell 5 ก่อน cell 11; ตัวแปรที่ยังไม่มี: " + ", ".join(missing))

if "prepared_yaml_path" not in globals():
    candidate_paths = []
    if "WORK_DIR" in globals():
        candidate_paths.append(WORK_DIR / "data.yaml")
    if "RAW_DIR" in globals():
        candidate_paths.append(RAW_DIR / "data.yaml")

    prepared_yaml_path = next((path for path in candidate_paths if path.exists()), None)
    if prepared_yaml_path is None:
        raise RuntimeError("กรุณารัน cell 9 ก่อน cell 11 เพื่อสร้าง data.yaml สำหรับ train")
    print(f"Status: recovered training yaml -> {prepared_yaml_path}")

DEVICE_MODE = "AUTO"  # เลือก "AUTO", "GPU" หรือ "CPU"
MODEL_NAME = "yolo11s.pt"


def resolve_training_device(mode: str):
    normalized_mode = mode.strip().upper()
    valid_modes = {"AUTO", "GPU", "CPU"}
    if normalized_mode not in valid_modes:
        raise ValueError(f"DEVICE_MODE ต้องเป็นหนึ่งใน {sorted(valid_modes)}")

    cuda_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if cuda_available else None

    if normalized_mode == "GPU":
        if not cuda_available:
            raise RuntimeError("ผู้ใช้เลือก GPU แต่ไม่พบ CUDA GPU ในเครื่องนี้")
        return 0, "GPU", gpu_name

    if normalized_mode == "CPU":
        return "cpu", "CPU", gpu_name

    if cuda_available:
        return 0, "GPU", gpu_name
    return "cpu", "CPU", gpu_name


resolved_device, resolved_device_label, detected_gpu_name = resolve_training_device(DEVICE_MODE)
TRAIN_ARGS = {
    "data": str(prepared_yaml_path),
    "epochs": 60,
    "batch": 16,
    "imgsz": 640,
    "project": str(BASE_DIR / "runs"),
    "name": "yolo11_fruit_detector",
    "exist_ok": True,
    "seed": SEED,
    "device": resolved_device,
    "degrees": 15.0,
    "translate": 0.10,
    "scale": 0.20,
    "fliplr": 0.50,
    "hsv_v": 0.20,
    "hsv_s": 0.20,
    "mosaic": 1.0,
    "mixup": 0.0,
    "plots": True,
}

augmentation_summary = (
    "หมุนภาพสูงสุด 15 องศา, เลื่อนภาพ 10%, ซูม 20%, "
    "สะท้อนซ้าย-ขวา 50%, ปรับความสว่างประมาณ +-20% (hsv_v=0.20), "
    "ปรับความอิ่มสี 20% (hsv_s=0.20), และ Mosaic"
)

device_summary = (
    f"**{resolved_device_label}**"
    if resolved_device_label == "CPU"
    else f"**{resolved_device_label}** ({detected_gpu_name})"
)

display(Markdown(
    "## 2. การตั้งค่าการเรียนรู้ (Training Setup)\n"
    f"- โหมดอุปกรณ์ที่เลือก: **{DEVICE_MODE}**\n"
    f"- อุปกรณ์ที่ใช้ train: {device_summary}\n"
    f"- CUDA พร้อมใช้งาน: **{torch.cuda.is_available()}**\n"
    f"- Augmentation ที่ใช้: **{augmentation_summary}**\n"
    f"- จำนวน Epochs: **{TRAIN_ARGS['epochs']}** รอบ | Batch Size: **{TRAIN_ARGS['batch']}**\n"
    f"- โมเดลตั้งต้น: **{MODEL_NAME}**\n"
    f"- โฟลเดอร์บันทึกผล: `{TRAIN_ARGS['project']}`"
))

print(f"Status: device mode = {DEVICE_MODE}")
if resolved_device_label == "GPU":
    print(f"Status: using GPU -> {detected_gpu_name} (device={resolved_device})")
else:
    print("Status: using CPU")


NameError: name 'prepared_yaml_path' is not defined

In [ ]:
from pathlib import Path

missing = [name for name in ("TRAIN_ARGS", "MODEL_NAME", "detected_gpu_name") if name not in globals()]
if missing:
    raise RuntimeError("กรุณารัน cell 11 ก่อน cell 12; ตัวแปรที่ยังไม่มี: " + ", ".join(missing))

print("Status: starting training...")
print(f"Status: training outputs will be saved to {TRAIN_ARGS['project']}")
if TRAIN_ARGS["device"] == "cpu":
    print("Status: confirmed training device -> CPU")
else:
    print(f"Status: confirmed training device -> GPU ({detected_gpu_name})")
    print(f"Status: CUDA device count = {torch.cuda.device_count()}")

model = YOLO(MODEL_NAME)
train_results = model.train(**TRAIN_ARGS)

run_dir = Path(TRAIN_ARGS["project"]) / TRAIN_ARGS["name"]
best_model_path = run_dir / "weights" / "best.pt"

print("Training เสร็จแล้ว")
print("Best model:", best_model_path)


Ultralytics 8.4.31  Python-3.13.5 torch-2.11.0+cpu CPU (AMD Ryzen 5 7600 6-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\NateZero\Documents\GitHub\CS483-Object-Detection-Masterclass\runtime_data\custom_data_yolo11\data.yaml, degrees=15.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.2, hsv_v=0.2, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11_fruit_detector, nb

In [ ]:
from IPython.display import Image

results_png = Path(TRAIN_ARGS["project"]) / TRAIN_ARGS["name"] / "results.png"
if results_png.exists():
    display(Image(filename=str(results_png)))
else:
    print("ยังไม่พบ results.png")


In [ ]:
best_model = YOLO(str(best_model_path))
metrics = best_model.val(
    data=str(prepared_yaml_path),
    split="test",
    imgsz=TRAIN_ARGS["imgsz"],
    batch=TRAIN_ARGS["batch"],
    plots=True,
)

map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)
gap = map50 - map50_95

if gap >= 0.20:
    gap_analysis = (
        "ค่า mAP@50-95 ต่ำกว่า mAP@50 มาก แปลว่าโมเดลมักตรวจพบวัตถุได้ แต่ตำแหน่งกรอบยังไม่แม่นพอเมื่อใช้ IoU ที่เข้มขึ้น "
        "สาเหตุที่เป็นไปได้คือวัตถุหลายคลาสมีลักษณะคล้ายกัน, ขอบวัตถุบางภาพไม่ชัด, หรือ annotation ของบางคลาสตีกรอบไม่สม่ำเสมอ"
    )
elif gap >= 0.10:
    gap_analysis = (
        "โมเดลเริ่มแยกคลาสได้ดีในระดับหนึ่ง แต่ความละเอียดของกรอบยังมีช่องว่างอยู่ "
        "ควรลองเพิ่มคุณภาพ annotation, เพิ่มจำนวนภาพต่อคลาสที่คล้ายกัน, หรือเพิ่มภาพที่มีมุมมองหลากหลาย"
    )
else:
    gap_analysis = (
        "ช่องว่างระหว่าง mAP@50 และ mAP@50-95 ไม่มาก แปลว่าทั้งการตรวจพบและการตีกรอบมีความสม่ำเสมอค่อนข้างดี"
    )

display(Markdown(
    "## 3. วิเคราะห์ผลลัพธ์เชิงสถิติ (Performance Analysis)\n"
    f"- ค่า mAP@50: **{map50:.4f}**\n"
    f"- ค่า mAP@50-95: **{map50_95:.4f}**\n"
    f"- วิเคราะห์ความต่าง: {gap_analysis}"
))


In [ ]:
import glob
from IPython.display import Image, display

test_images = sorted(glob.glob(str(prepared_dir / "test" / "images" / "*")))
sample_images = test_images[:5]

if not sample_images:
    print("No images found in test split")
else:
    prediction_results = best_model.predict(source=sample_images, conf=0.25, save=True, line_width=2)
    prediction_dir = Path(prediction_results[0].save_dir)

    for image_path in sorted(prediction_dir.glob("*"))[:5]:
        display(Image(filename=str(image_path), width=500))


## หมายเหตุสำหรับการส่งงาน

หลังจากรันครบทุก cell แล้ว นักศึกษาสามารถคัดลอกค่าจากผลลัพธ์ด้านบนไปสรุปในรายงานได้ทันที:

- จำนวนรูปภาพ: Train / Valid / Test
- Augmentation ที่ใช้
- จำนวน Epochs และ Batch Size
- ค่า `mAP@50` และ `mAP@50-95`
- เหตุผลว่าทำไม `mAP@50-95` อาจต่ำกว่า `mAP@50` มาก
